In [1]:
import pandas as pd
import anndata as ad
import h5py

In [2]:
df = pd.read_csv("../../Experiment/hme1_diaPASEF/Data/dia-quant-output/report.tsv", sep="\t", low_memory=False, nrows=500)
print(df.shape)
df.head(5)

(500, 71)


,File.Name,Run,Protein.Group,Protein.Ids,Protein.Names,Genes,PG.Quantity,PG.Normalised,PG.MaxLFQ,Genes.Quantity,...,All Mapped Genes,M:15.9949,M:15.9949 Best Localization,M:15.9949 Best Scan,STY:79.96633,STY:79.96633 Best Localization,STY:79.96633 Best Scan,n:42.0106,n:42.0106 Best Localization,n:42.0106 Best Scan
0,E:\FGCZ\p31978\Workunit_348536_20260722_103347...,20260702_028_C42386_S1171815_180,Q86U42,Q86U42,NaN,PABPN1,50749.593750,20912.267580,11466.715820,50749.593750,...,PABPN1,NaN,NaN,NaN,AAAAAAAAAAGAAGGRGS(1.0000)GPGR,1.0,20251127_003_S1060653_WT_starve_r1.238902.2389...,n(1.0000)AAAAAAAAAAGAAGGRGSGPGR,1.0,20251127_003_S1060653_WT_starve_r1.238902.2389...
1,E:\FGCZ\p31978\Workunit_348536_20260722_103347...,20260702_044_C42386_S1171804_94,Q86U42,Q86U42,NaN,PABPN1,7871.427734,4587.854004,11402.654300,7871.427734,...,PABPN1,NaN,NaN,NaN,AAAAAAAAAAGAAGGRGS(1.0000)GPGR,1.0,20251127_003_S1060653_WT_starve_r1.238902.2389...,n(1.0000)AAAAAAAAAAGAAGGRGSGPGR,1.0,20251127_003_S1060653_WT_starve_r1.238902.2389...
2,E:\FGCZ\p31978\Workunit_348536_20260722_103347...,20260702_062_C42386_S1171799_174,Q86U42,Q86U42,NaN,PABPN1,11014.580080,7947.322266,9037.083984,11014.580080,...,PABPN1,NaN,NaN,NaN,AAAAAAAAAAGAAGGRGS(1.0000)GPGR,1.0,20251127_003_S1060653_WT_starve_r1.238902.2389...,n(1.0000)AAAAAAAAAAGAAGGRGSGPGR,1.0,20251127_003_S1060653_WT_starve_r1.238902.2389...
3,E:\FGCZ\p31978\Workunit_348536_20260722_103347...,20260702_037_C42386_S1171814_153,Q86U42,Q86U42,NaN,PABPN1,27317.410160,15804.972660,9248.760742,27317.410160,...,PABPN1,NaN,NaN,NaN,AAAAAAAAAAGAAGGRGS(1.0000)GPGR,1.0,20251127_003_S1060653_WT_starve_r1.238902.2389...,n(1.0000)AAAAAAAAAAGAAGGRGSGPGR,1.0,20251127_003_S1060653_WT_starve_r1.238902.2389...
4,E:\FGCZ\p31978\Workunit_348536_20260722_103347...,20260702_033_C42386_S1171813_125,Q86U42,Q86U42,NaN,PABPN1,9870.259766,6694.289062,9399.364258,9870.259766,...,PABPN1,NaN,NaN,NaN,AAAAAAAAAAGAAGGRGS(1.0000)GPGR,1.0,20251127_003_S1060653_WT_starve_r1.238902.2389...,n(1.0000)AAAAAAAAAAGAAGGRGSGPGR,1.0,20251127_003_S1060653_WT_starve_r1.238902.2389...


# Exporting the diaPASEF precursor report to `.h5ad`

The DIA-NN `report.tsv` is a **long** table: one row per (run x precursor), with the run
annotation, the precursor annotation and the quantities all repeated on every row. `.h5ad`
(AnnData / [scverse](https://anndata.readthedocs.io)) is the opposite shape - one matrix plus
two annotation tables - so the export is a pivot, not a dump:

| AnnData slot | content here | size |
|---|---|---|
| `X` | **raw precursor intensities** (`Precursor.Quantity`, the un-normalised DIA-NN quantity) | runs x precursors |
| `layers[...]` | the same matrix for the other per-(run x precursor) quantities in the report (`Precursor.Normalised`, `Q.Value`, ...) | runs x precursors |
| `obs` | one row per **run** = one sample: file name, facility sample number and the decoded cell line / timepoint / replicate | runs x k |
| `var` | one row per **precursor**: modified and stripped sequence, charge, protein group, genes, PTM localisation strings | precursors x m |
| `uns` | provenance: source file, export date, what `X` is and what it is *not* | - |

Conventions this export follows, and the reasons:

- **`obs` = samples, `var` = features.** That is the scverse orientation (`n_obs x n_vars`), the
  one every reader of an `.h5ad` will assume. It is transposed relative to the project's own
  processed tables, where sites are rows.
- **A missing measurement is `NaN`, never 0.** A precursor absent from a run means *not detected /
  not quantified*, and 0 would read as "measured, zero intensity". Same rule as everywhere else in
  this project - no `.fillna(0)`.
- **`X` is raw.** No log, no normalisation, no imputation, no filtering beyond DIA-NN's own
  q-value control. That is what makes the file shareable: the recipient applies their own
  processing. `Precursor.Normalised` is carried alongside as a layer, so the DIA-NN normalisation
  is available without being imposed.
- **The matrix is dense `float32`.** Runs x precursors is a few hundred x a few hundred thousand,
  so dense costs a few hundred MB per layer, and sparse formats cannot represent `NaN` as
  "missing" - only 0, the one value we must not write.

## The size problem, and why this notebook streams

`report.tsv` is **~25 GB**. Reading it into a DataFrame first - what the obvious
`pd.read_csv(...)` -> `pivot_table(...)` route does - does not fit on this machine:

| | measured | 25 GB projection |
|---|---|---|
| `pd.read_csv` with `usecols` (18 of 71 columns) | 109 MB/s parse, **2.2 GB peak RAM per 1M rows** | ~4 min, **~40-50 GB RAM** |
| `pd.read_csv(chunksize=...)` | 175 MB/s | ~2.5 min, constant RAM, but you still have to assemble the matrix |
| **pyarrow streaming + incremental fill** (this notebook) | 140 MB/s end to end, peak **2.4 GB RAM** | **~3-6 min, ~3-5 GB RAM** |

*(Measured on this machine - M3 Pro, 36 GB - with a synthetic 1.47 GB / 1M-row report of the same
shape: 71 columns, 219 runs, 114k precursors, realistic string lengths.)*

At ~1100 bytes per row, 25 GB is roughly **20-25 million rows**. The killer is not the parsing, it
is that pandas stores each string cell as a Python object: 11 string columns x 20M rows is tens of
GB before any pivot happens. The output matrix, by contrast, is small - 219 runs x ~300k
precursors x 4 bytes = **~260 MB per layer**. The long form is what is big, not the data.

So the production path never materialises the long table:

1. `pyarrow.csv.open_csv` reads the file in 256 MB blocks, decoding **only the needed columns**.
2. Each block contributes (run index, precursor index, value) triplets, appended as compact numpy
   arrays; run and precursor dictionaries grow as new ones appear.
3. Annotation is captured **only on a key's first appearance**, so `obs` and `var` stay at 219 and
   ~300k rows instead of 20M.
4. After the last block the dense matrices are allocated once and filled by fancy indexing.

`long_report_to_anndata()` (the simple in-memory pivot) is kept for development passes on a
truncated read - `NROWS` in the config cell - and gives an identical object, so the streaming path
can be checked against it on a subset.

> **If a `report.parquet` sits next to the `.tsv`** (DIA-NN >= 2.0 writes one), prefer it: it is
> typed, columnar and several times faster to read - `pd.read_parquet(path, columns=KEEP_COLS)`
> reads only the columns it is asked for, without parsing the rest.

## Configuration

`STREAMING = True` is the production path (whole file, constant memory). Set `NROWS` to an integer
*and* `STREAMING = False` for a fast development pass over the first n rows.

`SAMPLE_MAP` decodes the facility sample number at the end of each `Run` name into the experimental
annotation, using the same numbering as `LFQ_diaPASEF.ipynb` (8 cell lines x 9 timepoints x 3
replicates, filled in that nesting order, plus the three mixed-reference runs).

⚠️ **Every entry of `LAYER_COLS` costs another full matrix** - ~260 MB in RAM and on disk before
compression. The default keeps the three that a recipient actually needs (DIA-NN's normalised
quantity, the MS1 area, and the run-specific q-value that says how trustworthy each cell is); add
`RT` / `IM` / `PEP` only if they will be used.

In [3]:
import os
from datetime import date
import time
import numpy as np
import pyarrow.csv as pv

REPORT_PATH = "../../Experiment/hme1_diaPASEF/Data/dia-quant-output/report.tsv"
OUT_DIR = "../../Experiment/hme1_diaPASEF/Data/Processed"
OUT_PATH = f"{OUT_DIR}/{date.today():%Y%m%d}_hme1_diaPASEF_precursor_raw_intensities.h5ad"

STREAMING = True        # True  -> pyarrow streaming, whole file, constant memory
NROWS = None            # int + STREAMING=False -> development pass over the first n rows only
BLOCK_SIZE_MB = 256     # streaming block size; larger = fewer, bigger blocks

RUN_COL = "Run"                    # one row of the matrix ... one sample
FEATURE_COL = "Precursor.Id"       # one column of the matrix: sequence + charge, unique per run
VALUE_COL = "Precursor.Quantity"   # what goes into X: the RAW intensity

# Same per-(run x precursor) shape as VALUE_COL, so each becomes a layer. Anything absent from the
# report is skipped rather than raising - DIA-NN's column set changes between versions.
LAYER_COLS = ["Precursor.Normalised",   # DIA-NN's own normalisation of VALUE_COL
              "Ms1.Area",               # MS1 precursor area
              "Q.Value",]               # run-specific precursor q-value
              # "Precursor.Translated", "PEP", "RT", "IM"   <- one matrix each, add if needed

# Per-run annotation (constant within a run).
OBS_COLS = ["File.Name",]

# Per-precursor annotation, taken from the first row in which that precursor appears.
VAR_COLS = ["Modified.Sequence", "Stripped.Sequence", "Precursor.Charge",
            "Protein.Group", "Protein.Ids", "Protein.Names", "Genes",
            "First.Protein.Description", "Proteotypic",
            "STY:79.96633", "STY:79.96633 Best Localization",
            "M:15.9949", "n:42.0106",]

KEEP_COLS = list(dict.fromkeys([RUN_COL, FEATURE_COL, VALUE_COL]
                               + LAYER_COLS + OBS_COLS + VAR_COLS))

# --- sample-number -> experimental annotation, as in LFQ_diaPASEF.ipynb ---------------------
CELL_LINES = ["WT", "EGFRT693A", "BRAFS151A1", "SOS1S1178A",
              "SHOC2T71A", "BRAFS151A2", "GAB1Y259A", "RPS6KA3S375A"]
TIME_POINTS = ["full", "starve", "2", "5", "10", "15", "20", "30", "90"]
REPLICATES = ["r1", "r2", "r3"]
CONDITION = "EGF"

SAMPLE_MAP = {}
_n = 1
for _cell in CELL_LINES:
    for _tp in TIME_POINTS:
        for _rep in REPLICATES:
            SAMPLE_MAP[str(_n)] = (_cell, _tp, _rep)
            _n += 1

# The three mixed-reference injections carry a token instead of a number.
SAMPLE_MAP.update({"mix":  ("MIX", "starve", "r1"),
                   "mixb": ("MIX", "starve", "r2"),
                   "mixc": ("MIX", "starve", "r3"),})

print(f"{len(SAMPLE_MAP)} sample codes mapped ({len(CELL_LINES)} cell lines x "
      f"{len(TIME_POINTS)} timepoints x {len(REPLICATES)} replicates + 3 mixes)")
print(f"reading {len(KEEP_COLS)} columns | mode: "
      f"{'streaming (full file)' if STREAMING else f'in-memory (NROWS={NROWS})'}")
if os.path.exists(REPORT_PATH):
    print(f"source: {REPORT_PATH}  ({os.path.getsize(REPORT_PATH)/1e9:,.1f} GB)")
print("output ->", OUT_PATH)

219 sample codes mapped (8 cell lines x 9 timepoints x 3 replicates + 3 mixes)
reading 20 columns | mode: streaming (full file)
source: ../../Experiment/hme1_diaPASEF/Data/dia-quant-output/report.tsv  (25.0 GB)
output -> ../../Experiment/hme1_diaPASEF/Data/Processed/20260819_hme1_diaPASEF_precursor_raw_intensities.h5ad


## Functions

Three pieces: the HDF5-safety cleanup for annotation tables, the two builders (streaming and
in-memory - same output, different memory profile), and the run-name decoder.

Details worth knowing about the builders:

- **Duplicated (run, precursor) pairs** should not exist - `Precursor.Id` already includes the
  charge - but both builders count them and resolve them by keeping the row with the **largest
  `VALUE_COL`** (layers follow that same row), rather than letting an arbitrary one win silently.
- **Annotation is captured only when a key is seen for the first time** - that is what keeps the
  streaming builder O(features) rather than O(rows). Within that block it takes the first
  *non-null* value, so a PTM string that is empty on a precursor's first row is still recovered;
  the report is written precursor by precursor, so a precursor's rows almost always share a block.
  The in-memory builder does the same over the whole frame. The two can therefore differ only for
  a precursor whose rows straddle a 256 MB block boundary **and** whose annotation is null in the
  earlier part - checked below, and in practice zero rows.
- **Object columns are cleaned before writing.** HDF5 has no missing-string concept, so string
  columns are filled with `""` and stored as `category` when they have few distinct values (cheap,
  and read back as a factor by both Python and R) or as plain strings otherwise. Annotation only -
  `X` keeps its `NaN`s.

In [4]:
def clean_frame_for_h5ad(frame,
                         max_categories=200,):
    """
    Make an annotation frame safe to write into an .h5ad file.

    HDF5 cannot store a missing string, and AnnData refuses mixed-type object columns, so every
    object / pandas-string column is filled with "" and cast either to `category` (few distinct
    values - compact, and read back as a factor by both Python and R readers) or to plain `str`.
    Numeric columns are left untouched, so numeric NaN survives.

    Args:
      frame: DataFrame to clean (an obs or var table); not modified in place
      max_categories: a string column with at most this many distinct values becomes a category

    Returns:
      A copy of the frame with all string-like columns written as category or str
    """
    out = frame.copy()
    for col in out.columns:
        if out[col].dtype == object or str(out[col].dtype) == "string":
            values = out[col].astype("object").where(out[col].notna(), "").astype(str)
            out[col] = values.astype("category") if values.nunique() <= max_categories else values
    return out


def _assemble_anndata(row_idx,
                      col_idx,
                      values,
                      value_col,
                      run_keys,
                      feature_keys,
                      obs,
                      var,
                      run_col,
                      feature_col,
                      dtype="float32",
                      verbose=True,):
    """
    Scatter (row, column, value) triplets into dense matrices and wrap them in an AnnData.

    Shared back end of the streaming and in-memory builders. Duplicated (row, column) pairs are
    counted and resolved by keeping the triplet with the largest `value_col` - the other layers
    follow that same triplet, so a cell is never assembled from two different rows of the report.

    Args:
      row_idx: int array, position of each triplet's run in `run_keys`
      col_idx: int array, position of each triplet's feature in `feature_keys`
      values: {column name: 1D array of values}, all aligned with row_idx / col_idx
      value_col: key of `values` that becomes X; the rest become layers
      run_keys: run names in matrix-row order
      feature_keys: feature names in matrix-column order
      obs: per-run annotation, indexed by run name (may be empty)
      var: per-feature annotation, indexed by feature name (may be empty)
      run_col: name given to the obs index
      feature_col: name given to the var index
      dtype: numeric dtype of X and of the layers
      verbose: print shape, duplicate count and missingness of X

    Returns:
      An AnnData with X = values[value_col], the remaining entries of `values` as layers, and the
      cleaned obs / var annotation
    """
    n_obs, n_var = len(run_keys), len(feature_keys)

    key = row_idx.astype(np.int64) * n_var + col_idx
    n_duplicated = int(len(key) - len(np.unique(key)))
    if n_duplicated:
        quantity = values[value_col]
        order = np.argsort(np.where(np.isnan(quantity), -np.inf, quantity), kind="stable")
        row_idx, col_idx = row_idx[order], col_idx[order]
        values = {name: array[order] for name, array in values.items()}
    del key

    matrices = {}
    for name, array in values.items():
        matrix = np.full((n_obs, n_var), np.nan, dtype=dtype)
        matrix[row_idx, col_idx] = array
        matrices[name] = matrix

    obs = obs.reindex(run_keys) if len(obs.columns) else pd.DataFrame(index=run_keys)
    var = var.reindex(feature_keys) if len(var.columns) else pd.DataFrame(index=feature_keys)
    obs.index = pd.Index([str(key) for key in run_keys], name=run_col)
    var.index = pd.Index([str(key) for key in feature_keys], name=feature_col)

    adata = ad.AnnData(X=matrices.pop(value_col),
                       obs=clean_frame_for_h5ad(obs),
                       var=clean_frame_for_h5ad(var),
                       layers=matrices,)
    if verbose:
        print(f"AnnData {adata.n_obs} runs x {adata.n_vars:,} features "
              f"| X = '{value_col}' | layers: {list(adata.layers)}")
        print(f"duplicated ({run_col}, {feature_col}) pairs: {n_duplicated:,}"
              + (" (resolved by keeping the largest value)" if n_duplicated else ""))
        print(f"missing values in X: {np.isnan(adata.X).mean():.1%} (kept as NaN, not zero)")
    return adata


def stream_report_to_anndata(path,
                             run_col,
                             feature_col,
                             value_col,
                             layer_cols=(),
                             obs_cols=(),
                             var_cols=(),
                             block_size_mb=256,
                             dtype="float32",
                             report_every=20,
                             verbose=True,):
    """
    Convert a long DIA-NN report into an AnnData (runs x features) without loading it into memory.

    The file is read in blocks with pyarrow, decoding only the requested columns. Each block
    contributes (run index, feature index, value) triplets stored as compact numpy arrays, and
    annotation is captured only the first time a run or a feature is seen - so memory scales with
    the number of runs and features, not with the number of rows. The dense matrices are allocated
    once, at the end. Measured at ~140 MB/s and ~2-5 GB peak RAM, i.e. a few minutes for a 25 GB
    report, against ~40-50 GB for the equivalent read-then-pivot.

    Missing (run, feature) combinations stay NaN - they mean "not detected", never zero.

    Args:
      path: path to the long report (.tsv, tab separated)
      run_col: column identifying the run / sample -> becomes obs_names
      feature_col: column identifying the feature (precursor) -> becomes var_names
      value_col: column whose values fill X (here: the raw precursor intensity)
      layer_cols: further per-(run x feature) columns, each stored as a layer of the same shape;
                  names absent from the file are skipped. Each costs one full matrix
      obs_cols: per-run annotation columns, first non-null value in the block where the run first
                appears
      var_cols: per-feature annotation columns, first non-null value in the block where the feature
                first appears
      block_size_mb: size of the blocks pyarrow decodes at a time
      dtype: numeric dtype of X and of the layers
      report_every: print a progress line every n blocks (0 to silence)
      verbose: print progress and the final summary

    Returns:
      An AnnData with X = value_col, the requested layers, and the obs / var annotation tables
    """
    with pv.open_csv(path,
                     read_options=pv.ReadOptions(block_size=block_size_mb << 20),
                     parse_options=pv.ParseOptions(delimiter="\t"),) as probe:
        available = set(probe.schema.names)

    missing_required = [col for col in (run_col, feature_col, value_col) if col not in available]
    if missing_required:
        raise KeyError(f"{path} is missing required column(s): {missing_required}")

    layer_cols = [col for col in layer_cols if col in available]
    obs_cols = [col for col in obs_cols if col in available]
    var_cols = [col for col in var_cols if col in available]
    wanted = list(dict.fromkeys([run_col, feature_col, value_col]
                                + layer_cols + obs_cols + var_cols))
    if verbose:
        skipped = [col for col in ((run_col, feature_col, value_col)
                                   + tuple(layer_cols) + tuple(obs_cols) + tuple(var_cols))
                   if col not in available]
        print(f"decoding {len(wanted)} of {len(available)} columns"
              + (f" | not in this report, skipped: {skipped}" if skipped else ""))

    run_index, feature_index = {}, {}
    row_parts, col_parts = [], []
    value_parts = {name: [] for name in [value_col] + layer_cols}
    obs_parts, var_parts = [], []
    n_rows = 0
    started = time.time()

    # strings_can_be_null=True makes an empty text field read as null. It is not pyarrow's
    # default (it would give the string "") but it is what pandas does, and it is what lets an
    # empty PTM field be recognised as "no annotation" instead of being stored as an empty string.
    with pv.open_csv(path,
                     read_options=pv.ReadOptions(block_size=block_size_mb << 20),
                     parse_options=pv.ParseOptions(delimiter="\t"),
                     convert_options=pv.ConvertOptions(include_columns=wanted,
                                                       strings_can_be_null=True,),) as reader:
        for n_block, block in enumerate(reader, start=1):
            chunk = block.to_pandas()
            n_rows += len(chunk)

            runs = chunk[run_col].to_numpy()
            features = chunk[feature_col].to_numpy()
            new_runs = pd.unique(runs[~pd.Series(runs).isin(run_index).to_numpy()])
            for key in new_runs:
                run_index[key] = len(run_index)
            new_features = pd.unique(features[~pd.Series(features).isin(feature_index).to_numpy()])
            for key in new_features:
                feature_index[key] = len(feature_index)

            row_parts.append(pd.Series(runs).map(run_index).to_numpy(np.int32))
            col_parts.append(pd.Series(features).map(feature_index).to_numpy(np.int32))
            for name in value_parts:
                value_parts[name].append(chunk[name].to_numpy(dtype))

            # Annotation only for keys seen for the first time - this is what keeps obs and var at
            # (runs) and (features) rows instead of growing with the file. `groupby(...).first()`
            # skips NaN, so a column that is null on a precursor's first row but filled on a later
            # one is still recovered, as long as both rows fall in the same block (the report is
            # written precursor by precursor, so they normally do).
            if len(new_runs) and obs_cols:
                fresh = chunk.loc[pd.Series(runs).isin(set(new_runs)).to_numpy(),
                                  [run_col] + obs_cols]
                obs_parts.append(fresh.groupby(run_col)[obs_cols].first())
            if len(new_features) and var_cols:
                fresh = chunk.loc[pd.Series(features).isin(set(new_features)).to_numpy(),
                                  [feature_col] + var_cols]
                var_parts.append(fresh.groupby(feature_col)[var_cols].first())

            if verbose and report_every and n_block % report_every == 0:
                print(f"  block {n_block}: {n_rows:,} rows | {len(run_index)} runs | "
                      f"{len(feature_index):,} features | {time.time() - started:,.0f}s")

    row_idx = np.concatenate(row_parts); del row_parts
    col_idx = np.concatenate(col_parts); del col_parts
    values = {name: np.concatenate(parts) for name, parts in value_parts.items()}
    del value_parts

    if verbose:
        print(f"read {n_rows:,} rows in {time.time() - started:,.0f}s "
              f"({os.path.getsize(path) / 1e6 / max(time.time() - started, 1):,.0f} MB/s)")

    return _assemble_anndata(row_idx=row_idx,
                             col_idx=col_idx,
                             values=values,
                             value_col=value_col,
                             run_keys=list(run_index),
                             feature_keys=list(feature_index),
                             obs=pd.concat(obs_parts) if obs_parts else pd.DataFrame(),
                             var=pd.concat(var_parts) if var_parts else pd.DataFrame(),
                             run_col=run_col,
                             feature_col=feature_col,
                             dtype=dtype,
                             verbose=verbose,)


def long_report_to_anndata(df,
                           run_col,
                           feature_col,
                           value_col,
                           layer_cols=(),
                           obs_cols=(),
                           var_cols=(),
                           dtype="float32",
                           verbose=True,):
    """
    Pivot an already-loaded long report into an AnnData (runs x features).

    Same output as `stream_report_to_anndata()`, for a frame that already fits in memory - a
    truncated development read, or a small dataset. Annotation is taken with
    `groupby(...).first()`, which skips NaN, so a column filled only in some runs is still
    recovered. Missing (run, feature) combinations stay NaN.

    Args:
      df: long DataFrame, one row per (run x feature)
      run_col: column identifying the run / sample -> becomes obs_names
      feature_col: column identifying the feature (precursor) -> becomes var_names
      value_col: column whose values fill X (here: the raw precursor intensity)
      layer_cols: further per-(run x feature) columns, each stored as a layer; names absent from
                  df are skipped
      obs_cols: per-run annotation columns, first non-null value per run
      var_cols: per-feature annotation columns, first non-null value per feature
      dtype: numeric dtype of X and of the layers
      verbose: print the shape, the duplicate count and the missingness of X

    Returns:
      An AnnData with X = value_col, the requested layers, and the obs / var annotation tables
    """
    missing_required = [col for col in (run_col, feature_col, value_col) if col not in df.columns]
    if missing_required:
        raise KeyError(f"frame is missing required column(s): {missing_required}")

    layer_cols = [col for col in layer_cols if col in df.columns]
    obs_cols = [col for col in obs_cols if col in df.columns]
    var_cols = [col for col in var_cols if col in df.columns]

    run_keys = list(pd.unique(df[run_col]))
    feature_keys = list(pd.unique(df[feature_col]))
    run_index = {key: i for i, key in enumerate(run_keys)}
    feature_index = {key: i for i, key in enumerate(feature_keys)}

    values = {name: df[name].to_numpy(dtype) for name in [value_col] + layer_cols}
    obs = df.groupby(run_col)[obs_cols].first() if obs_cols else pd.DataFrame()
    var = df.groupby(feature_col)[var_cols].first() if var_cols else pd.DataFrame()

    return _assemble_anndata(row_idx=df[run_col].map(run_index).to_numpy(np.int32),
                             col_idx=df[feature_col].map(feature_index).to_numpy(np.int32),
                             values=values,
                             value_col=value_col,
                             run_keys=run_keys,
                             feature_keys=feature_keys,
                             obs=obs,
                             var=var,
                             run_col=run_col,
                             feature_col=feature_col,
                             dtype=dtype,
                             verbose=verbose,)


def add_sample_annotation(adata,
                          sample_map,
                          condition=CONDITION,
                          verbose=True,):
    """
    Decode the facility sample number at the end of each run name into obs annotation.

    Run names look like '20260702_028_C42386_S1171815_180', where the last underscore-separated
    field is the sample number used when the samples were submitted ('mix' / 'mixb' / 'mixc' for
    the mixed reference injections). `sample_map` translates it into cell line, timepoint and
    replicate; runs whose code is not in the map get NaN and are reported.

    Args:
      adata: AnnData whose obs_names are the DIA-NN run names; modified in place
      sample_map: {sample code (str): (cell_line, timepoint, replicate)}
      condition: stimulation applied to every sample of this dataset (a constant column)
      verbose: print how many runs were decoded and list any that were not

    Returns:
      The same AnnData, with sample_code / cell_line / timepoint / replicate / condition and a
      project-convention `sample_name` column added to obs
    """
    codes = [str(run).split("_")[-1] for run in adata.obs_names]
    decoded = [sample_map.get(code, (np.nan, np.nan, np.nan)) for code in codes]

    adata.obs["sample_code"] = codes
    adata.obs["cell_line"] = [entry[0] for entry in decoded]
    adata.obs["timepoint"] = [entry[1] for entry in decoded]
    adata.obs["replicate"] = [entry[2] for entry in decoded]
    adata.obs["condition"] = condition

    # The name this run carries in the project's processed tables, so the two can be joined.
    adata.obs["sample_name"] = [f"{entry[0]}_raw:abs_{condition}_{entry[1]}_{entry[2]}"
                                if isinstance(entry[0], str) else np.nan for entry in decoded]

    unmatched = [run for run, entry in zip(adata.obs_names, decoded)
                 if not isinstance(entry[0], str)]
    if verbose:
        print(f"{adata.n_obs - len(unmatched)} of {adata.n_obs} runs decoded into "
              f"cell line / timepoint / replicate")
        if unmatched:
            print(f"unmatched run(s) ({len(unmatched)}):", unmatched[:10])
    adata.obs = clean_frame_for_h5ad(adata.obs)
    return adata

## Building the object

This is the long step - a few minutes on the full file. The progress lines report rows, runs and
features seen so far, so a stall is visible rather than silent.

In [5]:
started = time.time()

if STREAMING:
    adata = stream_report_to_anndata(REPORT_PATH,
                                     run_col=RUN_COL,
                                     feature_col=FEATURE_COL,
                                     value_col=VALUE_COL,
                                     layer_cols=LAYER_COLS,
                                     obs_cols=OBS_COLS,
                                     var_cols=VAR_COLS,
                                     block_size_mb=BLOCK_SIZE_MB,
                                     dtype="float32",
                                     verbose=True,)
else:
    # Development pass: read the first NROWS rows, then pivot in memory.
    report = pd.read_csv(REPORT_PATH,
                         sep="\t",
                         usecols=lambda col: col in set(KEEP_COLS),
                         low_memory=False,
                         nrows=NROWS,)
    print(f"{report.shape[0]:,} rows x {report.shape[1]} columns read "
          f"(NROWS={NROWS}) | {report[RUN_COL].nunique()} runs, "
          f"{report[FEATURE_COL].nunique():,} precursors")
    adata = long_report_to_anndata(report,
                                   run_col=RUN_COL,
                                   feature_col=FEATURE_COL,
                                   value_col=VALUE_COL,
                                   layer_cols=LAYER_COLS,
                                   obs_cols=OBS_COLS,
                                   var_cols=VAR_COLS,
                                   dtype="float32",
                                   verbose=True,)

adata = add_sample_annotation(adata,
                              sample_map=SAMPLE_MAP,
                              condition=CONDITION,
                              verbose=True,)

# Runs in submission order rather than in the order they happened to appear in the file.
adata = adata[np.argsort([int(code) if code.isdigit() else 10**6
                          for code in adata.obs["sample_code"]], kind="stable")].copy()

print(f"\ntotal {time.time() - started:,.0f}s | "
      f"matrix footprint {adata.X.nbytes * (1 + len(adata.layers)) / 1e9:,.2f} GB in memory")
adata

decoding 19 of 71 columns
  block 20: 3,484,232 rows | 219 runs | 44,308 features | 41s
  block 40: 7,013,313 rows | 219 runs | 90,532 features | 66s
  block 60: 10,523,282 rows | 219 runs | 138,539 features | 92s
  block 80: 13,968,053 rows | 219 runs | 184,653 features | 117s
read 16,299,791 rows in 121s (207 MB/s)
AnnData 219 runs x 215,647 features | X = 'Precursor.Quantity' | layers: ['Precursor.Normalised', 'Ms1.Area', 'Q.Value']
duplicated (Run, Precursor.Id) pairs: 0
missing values in X: 65.5% (kept as NaN, not zero)
219 of 219 runs decoded into cell line / timepoint / replicate

total 126s | matrix footprint 0.76 GB in memory


AnnData object with n_obs × n_vars = 219 × 215647
    obs: 'File.Name', 'sample_code', 'cell_line', 'timepoint', 'replicate', 'condition', 'sample_name'
    var: 'Modified.Sequence', 'Stripped.Sequence', 'Precursor.Charge', 'Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'Proteotypic', 'STY:79.96633', 'STY:79.96633 Best Localization', 'M:15.9949', 'n:42.0106'
    layers: 'Precursor.Normalised', 'Ms1.Area', 'Q.Value'

### Sanity checks before writing

Four checks, each aimed at a way this export could be wrong without looking wrong: the design is
complete (every cell line x timepoint x replicate present once), the matrix agrees cell by cell
with the file it came from, no zero was invented where a value was missing, and the detected
count per run is in the expected range.

In [6]:
# 1. Design completeness - one run per cell line x timepoint x replicate.
design = (adata.obs.groupby(["cell_line", "timepoint"], observed=True)
                   .size().unstack(fill_value=0))
design = design.reindex(index=[c for c in CELL_LINES + ["MIX"] if c in design.index],
                        columns=[t for t in TIME_POINTS if t in design.columns],)
print("runs per cell line x timepoint (expected 3 for every real sample):")
display(design)

# 2. The matrix reproduces the source file. A small probe read straight from disk, so this stays
#    an independent check of the streaming path rather than of the object against itself.
probe = pd.read_csv(REPORT_PATH,
                    sep="\t",
                    usecols=lambda col: col in {RUN_COL, FEATURE_COL, VALUE_COL},
                    low_memory=False,
                    nrows=200_000,).sample(8, random_state=0)

print("\nspot check, matrix vs report rows read directly from disk:")
for _, row in probe.iterrows():
    got = adata[str(row[RUN_COL]), str(row[FEATURE_COL])].X[0, 0]
    expected = row[VALUE_COL]
    ok = (np.isnan(got) and pd.isna(expected)) or np.isclose(got, expected, rtol=1e-6)
    print(f"  {'OK ' if ok else 'MISMATCH'} {str(row[FEATURE_COL])[:38]:<38} "
          f"{str(row[RUN_COL])[-12:]:>12}  {got:,.1f} vs {expected:,.1f}")

# 3. No missing value was turned into a zero.
print(f"\nexact zeros in X: {int((adata.X == 0).sum())} "
      f"(expected 0 - a non-detection must stay NaN)")

# 4. Precursors quantified per run - the number that should look like a DIA phospho run, and the
#    first place a truncated read or a mis-parsed block would show up.
per_run = np.isfinite(adata.X).sum(axis=1)
print(f"precursors quantified per run: median {np.median(per_run):,.0f} | "
      f"min {per_run.min():,} | max {per_run.max():,}")
print(f"precursors quantified in every run: {int((np.isfinite(adata.X).all(axis=0)).sum()):,} "
      f"of {adata.n_vars:,}")

runs per cell line x timepoint (expected 3 for every real sample):


timepoint,full,starve,2,5,10,15,20,30,90
cell_line,,,,,,,,,
WT,3,3,3,3,3,3,3,3,3
EGFRT693A,3,3,3,3,3,3,3,3,3
BRAFS151A1,3,3,3,3,3,3,3,3,3
SOS1S1178A,3,3,3,3,3,3,3,3,3
SHOC2T71A,3,3,3,3,3,3,3,3,3
BRAFS151A2,3,3,3,3,3,3,3,3,3
GAB1Y259A,3,3,3,3,3,3,3,3,3
RPS6KA3S375A,3,3,3,3,3,3,3,3,3
MIX,0,3,0,0,0,0,0,0,0



spot check, matrix vs report rows read directly from disk:
  OK  (UniMod:1)AEDEPDAKS(UniMod:21)PK2      _S1171327_45  1,637.1 vs 1,637.1
  OK  (UniMod:1)ATSPQKS(UniMod:21)PS(UniMod: 1171841_mixc  1,683.0 vs 1,683.0
  OK  (UniMod:1)AEPPS(UniMod:21)PVHC(UniMod: S1171718_170  972.0 vs 972.0
  OK  (UniMod:1)ATDT(UniMod:21)SQGELVHPK2    S1171757_158  2,555.1 vs 2,555.1
  OK  (UniMod:1)AGVEEVAASGSHLNGDLDPDDREEGAAS S1171324_175  1,898.1 vs 1,898.1
  OK  (UniMod:1)AAAVAAPLAAGGEEAAAT(UniMod:21 _S1171320_69  2,192.1 vs 2,192.1
  OK  (UniMod:1)AAAAAAAGDS(UniMod:21)DS(UniM _S1171705_32  6,319.2 vs 6,319.2
  OK  (UniMod:1)AAAMDVDT(UniMod:21)PSGTNSGAG _S1171770_28  6,863.3 vs 6,863.3

exact zeros in X: 183 (expected 0 - a non-detection must stay NaN)
precursors quantified per run: median 74,939 | min 1,321 | max 98,442
precursors quantified in every run: 346 of 215,647


### Provenance

Whoever receives the file should not have to ask what `X` is. This travels inside it.

In [7]:
adata.uns["dataset"] = "hme1_diaPASEF"
adata.uns["description"] = ("hTERT-HME1 wild type and 7 MAPK/ERK pathway mutant cell lines, EGF "
                            "time course (0.157 nM EGF), phosphoproteomics, LFQ diaPASEF, "
                            "3 replicates.")
adata.uns["source_file"] = os.path.basename(REPORT_PATH)
adata.uns["search_engine"] = "DIA-NN (FragPipe diaPASEF workflow)"
adata.uns["X"] = (f"{VALUE_COL}: raw precursor intensity as reported, no normalisation, no log "
                  f"transformation, no imputation. NaN = not detected in that run (NOT zero).")
adata.uns["layers"] = {name: f"per-(run x precursor) value of the report column '{name}'"
                       for name in adata.layers}
adata.uns["obs_names"] = "DIA-NN run name; the trailing field is the submitted sample number"
adata.uns["var_names"] = "DIA-NN Precursor.Id (modified sequence + charge)"
adata.uns["exported"] = f"{date.today():%Y-%m-%d}"

adata.uns

OrderedDict([('dataset', 'hme1_diaPASEF'),
             ('description',
              'hTERT-HME1 wild type and 7 MAPK/ERK pathway mutant cell lines, EGF time course (0.157 nM EGF), phosphoproteomics, LFQ diaPASEF, 3 replicates.'),
             ('source_file', 'report.tsv'),
             ('search_engine', 'DIA-NN (FragPipe diaPASEF workflow)'),
             ('X',
              'Precursor.Quantity: raw precursor intensity as reported, no normalisation, no log transformation, no imputation. NaN = not detected in that run (NOT zero).'),
             ('layers',
              {'Precursor.Normalised': "per-(run x precursor) value of the report column 'Precursor.Normalised'",
               'Ms1.Area': "per-(run x precursor) value of the report column 'Ms1.Area'",
               'Q.Value': "per-(run x precursor) value of the report column 'Q.Value'"}),
             ('obs_names',
              'DIA-NN run name; the trailing field is the submitted sample number'),
             ('var_names', 'DI

## Writing the file

Gzip compression is applied because the matrix is largely NaN and the annotation is highly
repetitive; it shrinks the file several-fold at no cost to the reader. Expect roughly
`runs x precursors x 4 bytes x (1 + n_layers)` before compression - about 1 GB for 219 x 300k with
three layers - and a few tens of seconds to write. The filename is date-stamped and the write
refuses to overwrite, so a file that has already been shared cannot be silently replaced.

In [9]:
os.makedirs(OUT_DIR, exist_ok=True)
if os.path.exists(OUT_PATH):
    raise FileExistsError(f"{OUT_PATH} already exists - it may already have been shared. "
                          f"Rename the output or delete it deliberately before re-writing.")

started = time.time()
adata.write_h5ad(OUT_PATH,
                 compression="gzip",)

print(f"written: {OUT_PATH}")
print(f"{os.path.getsize(OUT_PATH) / 1e9:,.2f} GB in {time.time() - started:,.0f}s "
      f"(uncompressed matrices: {adata.X.nbytes * (1 + len(adata.layers)) / 1e9:,.2f} GB)")

written: ../../Experiment/hme1_diaPASEF/Data/Processed/20260819_hme1_diaPASEF_precursor_raw_intensities.h5ad
0.31 GB in 17s (uncompressed matrices: 0.76 GB)


### Reading it back

The check that matters for a file that leaves the building: reopen it as a stranger would and
confirm the matrix, the annotation and the layers survived the round trip.

In [10]:
check = ad.read_h5ad(OUT_PATH)

print(check)
print("\nX identical after round trip:",
      np.array_equal(adata.X, check.X, equal_nan=True))
print("layers identical:",
      all(np.array_equal(adata.layers[k], check.layers[k], equal_nan=True) for k in adata.layers))
print("obs columns:", list(check.obs.columns))
print("var columns:", list(check.var.columns))
print("\nX =", check.uns["X"])

display(check.obs.head(3))
display(check.var.head(3))

AnnData object with n_obs × n_vars = 219 × 215647
    obs: 'File.Name', 'sample_code', 'cell_line', 'timepoint', 'replicate', 'condition', 'sample_name'
    var: 'Modified.Sequence', 'Stripped.Sequence', 'Precursor.Charge', 'Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'Proteotypic', 'STY:79.96633', 'STY:79.96633 Best Localization', 'M:15.9949', 'n:42.0106'
    uns: 'X', 'dataset', 'description', 'exported', 'layers', 'obs_names', 'search_engine', 'source_file', 'var_names'
    layers: 'Ms1.Area', 'Precursor.Normalised', 'Q.Value'

X identical after round trip: True
layers identical: True
obs columns: ['File.Name', 'sample_code', 'cell_line', 'timepoint', 'replicate', 'condition', 'sample_name']
var columns: ['Modified.Sequence', 'Stripped.Sequence', 'Precursor.Charge', 'Protein.Group', 'Protein.Ids', 'Protein.Names', 'Genes', 'Proteotypic', 'STY:79.96633', 'STY:79.96633 Best Localization', 'M:15.9949', 'n:42.0106']

X = Precursor.Quantity: raw precursor intensity as report

,File.Name,sample_code,cell_line,timepoint,replicate,condition,sample_name
Run,,,,,,,
20260619_060_C42386_S1171286_1,E:\FGCZ\p31978\Workunit_348536_20260722_103347...,1,WT,full,r1,EGF,WT_raw:abs_EGF_full_r1
20260625_076_C42386_S1171696_2,E:\FGCZ\p31978\Workunit_348536_20260722_103347...,2,WT,full,r2,EGF,WT_raw:abs_EGF_full_r2
20260702_032_C42386_S1171769_3,E:\FGCZ\p31978\Workunit_348536_20260722_103347...,3,WT,full,r3,EGF,WT_raw:abs_EGF_full_r3


,Modified.Sequence,Stripped.Sequence,Precursor.Charge,Protein.Group,Protein.Ids,Protein.Names,Genes,Proteotypic,STY:79.96633,STY:79.96633 Best Localization,M:15.9949,n:42.0106
Precursor.Id,,,,,,,,,,,,
(UniMod:1)AAAAAAAAAAGAAGGRGS(UniMod:21)GPGR2,(UniMod:1)AAAAAAAAAAGAAGGRGS(UniMod:21)GPGR,AAAAAAAAAAGAAGGRGSGPGR,2,Q86U42,Q86U42,,PABPN1,1,AAAAAAAAAAGAAGGRGS(1.0000)GPGR,1.0000,,n(1.0000)AAAAAAAAAAGAAGGRGSGPGR
(UniMod:1)AAAAAAAAAAGAAGGRGSGPGR2,(UniMod:1)AAAAAAAAAAGAAGGRGSGPGR,AAAAAAAAAAGAAGGRGSGPGR,2,Q86U42,Q86U42,,PABPN1,1,,NaN,,n(1.0000)AAAAAAAAAAGAAGGRGSGPGR
(UniMod:1)AAAAAAAGAAGS(UniMod:21)AAPAAAAGAPGS(UniMod:21)GGAPS(UniMod:21)GSQGVLIGDR3,(UniMod:1)AAAAAAAGAAGS(UniMod:21)AAPAAAAGAPGS(...,AAAAAAAGAAGSAAPAAAAGAPGSGGAPSGSQGVLIGDR,3,Q96S94,Q96S94,,CCNL2,1,AAAAAAAGAAGS(0.7707)AAPAAAAGAPGS(0.7431)GGAPS(...,0.7707,,n(1.0000)AAAAAAAGAAGSAAPAAAAGAPGSGGAPSGSQGVLIGDR


## Notes for whoever receives the file

```python
import anndata as ad
adata = ad.read_h5ad("20260819_hme1_diaPASEF_precursor_raw_intensities.h5ad")

adata.X                      # raw precursor intensities, runs x precursors, NaN = not detected
adata.obs                    # sample annotation: cell line, timepoint, replicate, condition
adata.var                    # precursor annotation: sequences, charge, protein, genes, PTM strings
adata.layers["Precursor.Normalised"]   # DIA-NN's normalised quantity, same shape
adata.layers["Q.Value"]                # run-specific q-value of each quantified cell

wt = adata[adata.obs["cell_line"] == "WT"]           # subset by sample annotation
sty = adata[:, adata.var["STY:79.96633"] != ""]      # phospho-localised precursors only
```

`ad.read_h5ad(path, backed="r")` opens it without loading `X`, which is worth knowing if the
recipient only wants the annotation.

In R the same file opens with `zellkonverter::readH5AD()` (-> `SingleCellExperiment`) or
`anndata::read_h5ad()`.

Two things the recipient must be told, because no file format can enforce them:

- **`X` is raw and unnormalised.** No sample-loading normalisation has been applied anywhere - the
  run medians of this dataset differ by up to ~2-fold - so any comparison across runs needs
  normalising first.
- **`NaN` means not detected, not zero.** Replacing it with 0 before log-transforming is the
  single most common way to ruin a DIA phosphoproteomics matrix.